In [3]:
import os
import sqlite3
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

class SoccerAnalyzer:
    def __init__(self, league_id):
        """Initialiseert de verbinding met de database."""
        path = kagglehub.dataset_download("hugomathien/soccer")
        self.db_path = os.path.join(path, 'database.sqlite')
        self.conn = sqlite3.connect(self.db_path)
        print(f"Verbonden met database op: {self.db_path}")

        self.league_id = league_id

    def get_matches_per_season(self, team_id):
        """1A: Toont het aantal wedstrijden per seizoen."""
        query = f"""
        SELECT season, COUNT(*) as aantal_wedstrijden
        FROM Match
        WHERE home_team_api_id = {team_id} OR away_team_api_id = {team_id}
        GROUP BY season
        """
        return pd.read_sql(query, self.conn)
    
    def get_matches_2010(self, team_id):
        """
        1B: Toont het totaal aantal wedstrijden in het kalenderjaar 2010, 
        onderverdeeld per voetbalseizoen.
        """
        query = f"""
        SELECT 
            season AS Seizoen, 
            COUNT(*) AS aantal_wedstrijden
        FROM Match
        WHERE (home_team_api_id = {team_id} OR away_team_api_id = {team_id})
        AND date BETWEEN '2010-01-01' AND '2010-12-31'
        GROUP BY season
        """
        return pd.read_sql(query, self.conn) 
    
    def get_league_standings(self):
        """
        1C & 1D: Berekent punten en ranglijst. 
        VOEGT team_id TOE voor de merge in stap 2a.
        """
        query = f"""
        WITH MatchPoints AS (
            SELECT 
                season, home_team_api_id AS team_id,
                CASE WHEN home_team_goal > away_team_goal THEN 3
                     WHEN home_team_goal = away_team_goal THEN 1 ELSE 0 END AS points -- Bereken de punten voor elke match
                     -- 3 punten voor een win, 1 punt voor een gelijkspel, 0 punten voor een verlies
            
            FROM Match WHERE league_id = {self.league_id} -- Pak de matches van de gekozen competitie
            
            UNION ALL
            
            SELECT 
                season, away_team_api_id AS team_id,
                CASE WHEN away_team_goal > home_team_goal THEN 3
                     WHEN away_team_goal = home_team_goal THEN 1 ELSE 0 END AS points
            FROM Match WHERE league_id = {self.league_id}
        )
        SELECT 
            MP.season AS Seizoen, 
            MP.team_id,
            T.team_long_name AS Teamnaam, 
            SUM(MP.points) AS Totaal_Punten,
            RANK() OVER (PARTITION BY MP.season ORDER BY SUM(MP.points) DESC) AS Ranglijst_Positie
        FROM MatchPoints MP
        JOIN Team T ON T.team_api_id = MP.team_id
        GROUP BY MP.season, MP.team_id, T.team_long_name -- Ook hier toevoegen
        ORDER BY MP.season, Ranglijst_Positie
        """

        return pd.read_sql(query, self.conn)

    def get_my_team_ranking_per_season(self, all_standings, team_name):
        """
        1D: Toont de eindpositie van jouw team op basis van de ranglijst uit 1C.
        """
        # We pakken het resultaat van 1C en filteren op jouw team
        my_team_ranking = all_standings[all_standings['Teamnaam'] == team_name]
        
        return my_team_ranking[['Seizoen', 'Teamnaam', 'Totaal_Punten', 'Ranglijst_Positie']]

# Voer uit
analyzer = SoccerAnalyzer(21518)
display(analyzer.get_matches_per_season(8634))
display(analyzer.get_matches_2010(8634))
display(analyzer.get_league_standings())
display(analyzer.get_my_team_ranking_per_season(analyzer.get_league_standings(), 'FC Barcelona'))

Verbonden met database op: C:\Users\yorri\.cache\kagglehub\datasets\hugomathien\soccer\versions\10\database.sqlite


,season,aantal_wedstrijden
0,2008/2009,38
1,2009/2010,38
2,2010/2011,38
3,2011/2012,38
4,2012/2013,38
5,2013/2014,38
6,2014/2015,38
7,2015/2016,38


,Seizoen,aantal_wedstrijden
0,2009/2010,23
1,2010/2011,16


,Seizoen,team_id,Teamnaam,Totaal_Punten,Ranglijst_Positie
0,2008/2009,8634,FC Barcelona,87,1
1,2008/2009,8633,Real Madrid CF,78,2
2,2008/2009,8302,Sevilla FC,70,3
3,2008/2009,9906,Atlético Madrid,67,4
4,2008/2009,10205,Villarreal CF,65,5
...,...,...,...,...,...
155,2015/2016,7878,Granada CF,39,16
156,2015/2016,9869,Real Sporting de Gijón,39,16
157,2015/2016,8370,Rayo Vallecano,38,18
158,2015/2016,8305,Getafe CF,36,19


,Seizoen,Teamnaam,Totaal_Punten,Ranglijst_Positie
0,2008/2009,FC Barcelona,87,1
20,2009/2010,FC Barcelona,99,1
40,2010/2011,FC Barcelona,96,1
61,2011/2012,FC Barcelona,91,2
80,2012/2013,FC Barcelona,100,1
102,2013/2014,FC Barcelona,87,2
120,2014/2015,FC Barcelona,94,1
140,2015/2016,FC Barcelona,91,1
